> # ⚠️ Under Construction
>
> **This notebook is currently broken and under construction.** The MCP server connection path is not yet working reliably and is being actively worked on. Expect errors if you run the cells below.
>
> Treat this notebook as a preview only. For working graph queries, use the direct Neo4j path in notebooks `03`, `04`, and `05`.

# Querying Neo4j via MCP (Model Context Protocol)

This notebook demonstrates how to query the same Neo4j knowledge graph from Labs 2-3 through an **MCP server** instead of a direct Neo4j driver connection.

**What is MCP?** The Model Context Protocol is an open standard for giving AI tools structured access to data sources. Instead of connecting directly to a database with credentials, your code talks to an MCP server that manages the connection.

**Prerequisites:** Complete [03 Data and Embeddings](03_data_and_embeddings.ipynb) first (creates the indexes and Document-Chunk structure this notebook queries).

**Learning Objectives:**
- Connect to a remote MCP server using the FastMCP client
- Discover available tools and retrieve the graph schema
- Execute Cypher queries via MCP's `read-cypher` tool
- Replicate the notebook 04 retrieval patterns (document context, adjacent chunks, topology traversal, operating limits)
- Understand how MCP decouples clients from database credentials

---

## Architecture

```
Direct (notebooks 04/05):
  Notebook ──> neo4j driver ──> Neo4j Aura
               (credentials)

MCP (this notebook):
  Notebook ──> FastMCP Client ──> MCP Server ──> Neo4j Aura
               (API key)          (manages DB credentials)
```

The MCP server handles authentication to Neo4j. Your notebook only needs the MCP endpoint URL and an API key.

## Install FastMCP

FastMCP provides a clean Pythonic client for the MCP protocol.

In [ ]:
%pip install fastmcp nest_asyncio typing_extensions>=4.12 -q
dbutils.library.restartPython()

## Configuration

Enter the MCP server connection details from your provided `MCP_ACCESS` config file.

In [ ]:
# ============================================
# CONFIGURATION - Enter your MCP server details
# ============================================

MCP_ENDPOINT = ""  # e.g., "https://neo4jmcp-app-dev.greenwater-82835120.eastus.azurecontainerapps.io"
MCP_API_KEY = ""   # Your API key from the MCP_ACCESS config
MCP_PATH = "/mcp"  # MCP path (usually /mcp)

# Validate configuration
if not MCP_ENDPOINT or not MCP_API_KEY:
    print("WARNING: Please enter your MCP connection details above before running the notebook!")
else:
    print("Configuration ready!")
    print(f"MCP URL: {MCP_ENDPOINT}{MCP_PATH}")

## Connect to the MCP Server

Create the FastMCP client with Bearer token authentication and verify connectivity.

In [ ]:
import asyncio
import nest_asyncio
from fastmcp import Client
from fastmcp.client.auth import BearerAuth

# Allow nested event loops (needed in Jupyter/Databricks)
nest_asyncio.apply()

MCP_URL = f"{MCP_ENDPOINT.rstrip('/')}{MCP_PATH}"
client = Client(MCP_URL, auth=BearerAuth(MCP_API_KEY))

print(f"Client created for: {MCP_URL}")

In [ ]:
async def mcp_call_tool(name: str, arguments: dict = None) -> str:
    """Call an MCP tool and return the text result."""
    async with client:
        result = await client.call_tool(name, arguments or {})
        # Extract text from result content
        if hasattr(result, 'content') and result.content:
            return result.content[0].text
        return str(result)

---

# Part 1: Discover and Explore

Every MCP server exposes a set of **tools** that clients can discover and call. This is the MCP equivalent of exploring an API's endpoints.

## List Available Tools

Discover what tools the MCP server provides.

In [ ]:
async def list_mcp_tools():
    async with client:
        tools = await client.list_tools()
        print(f"Available tools ({len(tools)}):\n")
        for tool in tools:
            print(f"  {tool.name}")
            print(f"    {tool.description[:100]}")
            print()

asyncio.run(list_mcp_tools())

## Get Graph Schema

The `get-schema` tool returns node labels, relationship types, and properties in your Neo4j database. In Lab 1, you explored this in the Neo4j Query Workspace. Here the MCP server provides it programmatically.

In [ ]:
schema = asyncio.run(mcp_call_tool("get-schema"))
print("Graph Schema:")
print("=" * 70)
print(schema)

## Graph Statistics

Run a Cypher query through MCP to count nodes by label.

In [ ]:
result = asyncio.run(mcp_call_tool("read-cypher", {
    "query": """
        MATCH (n)
        WITH labels(n) AS nodeLabels
        UNWIND nodeLabels AS label
        RETURN label, count(*) AS count
        ORDER BY count DESC
    """
}))
print("Node counts by label:\n")
print(result)

## Aircraft Fleet Overview

Query the aircraft topology that was loaded in Lab 2.

In [ ]:
result = asyncio.run(mcp_call_tool("read-cypher", {
    "query": """
        MATCH (a:Aircraft)-[:HAS_SYSTEM]->(s:System)
        RETURN a.tail_number AS aircraft, a.model AS model,
               collect(s.name) AS systems
        ORDER BY a.tail_number
        LIMIT 5
    """
}))
print("Aircraft Fleet:\n")
print(result)

---

# Part 2: Document-Chunk Queries

In notebook 03, we created a Document-Chunk structure from maintenance manuals. These queries explore that structure through MCP.

## Document Overview

List documents and their chunk counts.

In [ ]:
result = asyncio.run(mcp_call_tool("read-cypher", {
    "query": """
        MATCH (d:Document)
        OPTIONAL MATCH (d)<-[:FROM_DOCUMENT]-(c:Chunk)
        RETURN d.documentId AS document_id, d.title AS title,
               d.aircraftType AS aircraft_type, count(c) AS chunks
    """
}))
print("Documents in the graph:\n")
print(result)

## Chunk Chain Traversal

Walk the `NEXT_CHUNK` relationships created in notebook 03 to see how chunks are linked sequentially.

In [ ]:
result = asyncio.run(mcp_call_tool("read-cypher", {
    "query": """
        MATCH (c:Chunk)-[:FROM_DOCUMENT]->(d:Document)
        WHERE c.index IS NOT NULL
        OPTIONAL MATCH (c)-[:NEXT_CHUNK]->(next:Chunk)
        RETURN c.index AS chunk_index,
               substring(c.text, 0, 80) AS preview,
               next.index AS next_chunk
        ORDER BY c.index
        LIMIT 5
    """
}))
print("Chunk chain (first 5):\n")
print(result)

---

# Part 3: Graph-Enhanced Search via MCP

In notebook 04, the `VectorCypherRetriever` combined vector search with graph traversal behind a Python abstraction. Here you write the equivalent Cypher explicitly and send it through MCP's `read-cypher` tool.

We use **fulltext search** (`db.index.fulltext.queryNodes`) as the entry point instead of vector search, since the MCP client doesn't need an embedding model.

> **Note:** The `maintenanceChunkText` fulltext index was created in notebook 03.

## Example 1: Document Context Enrichment

Find relevant chunks via fulltext search, then traverse `FROM_DOCUMENT` to get source document metadata.

This mirrors notebook 04's `document_context_query` VectorCypherRetriever.

In [ ]:
result = asyncio.run(mcp_call_tool("read-cypher", {
    "query": """
        CALL db.index.fulltext.queryNodes('maintenanceChunkText', 'engine vibration troubleshoot')
        YIELD node, score
        MATCH (node)-[:FROM_DOCUMENT]->(doc:Document)
        RETURN doc.documentId AS document_id,
               doc.aircraftType AS aircraft_type,
               doc.title AS title,
               node.index AS chunk_index,
               score,
               substring(node.text, 0, 200) AS context
        ORDER BY score DESC
        LIMIT 3
    """
}))
print("Document-enriched search results:\n")
print(result)

**How it works:**
1. Fulltext search finds chunks matching the keywords
2. Graph traversal follows `FROM_DOCUMENT` to get document metadata
3. Results include document ID, aircraft type, title alongside the text

Compare to notebook 04's VectorCypherRetriever which does the same traversal but uses vector similarity (embeddings) as the entry point instead of keyword matching.

## Example 2: Adjacent Chunk Retrieval

Retrieve surrounding context by following `NEXT_CHUNK` relationships forward and backward.

This mirrors notebook 04's `adjacent_chunks_query` VectorCypherRetriever.

In [ ]:
result = asyncio.run(mcp_call_tool("read-cypher", {
    "query": """
        CALL db.index.fulltext.queryNodes('maintenanceChunkText', 'hydraulic pressure limits')
        YIELD node, score
        OPTIONAL MATCH (prev:Chunk)-[:NEXT_CHUNK]->(node)
        OPTIONAL MATCH (node)-[:NEXT_CHUNK]->(next:Chunk)
        MATCH (node)-[:FROM_DOCUMENT]->(doc:Document)
        RETURN doc.documentId AS document_id,
               node.index AS chunk_index,
               substring(COALESCE(prev.text, ''), 0, 100) AS previous_context,
               substring(node.text, 0, 200) AS main_context,
               substring(COALESCE(next.text, ''), 0, 100) AS next_context
        ORDER BY score DESC
        LIMIT 3
    """
}))
print("Adjacent chunk retrieval:\n")
print(result)

**Why this matters:** Maintenance procedures often span multiple chunks. By retrieving the previous and next chunks alongside the match, you get more complete procedure context. This is exactly what the `VectorCypherRetriever` does under the hood in notebook 04.

## Example 3: Aircraft Topology Traversal

Start from a search result, traverse through Document -> Aircraft -> System -> Component.

This mirrors notebook 04's `system_context_query` VectorCypherRetriever and demonstrates the full power of graph-enhanced retrieval.

In [ ]:
result = asyncio.run(mcp_call_tool("read-cypher", {
    "query": """
        CALL db.index.fulltext.queryNodes('maintenanceChunkText', 'fuel pump maintenance')
        YIELD node, score
        MATCH (node)-[:FROM_DOCUMENT]->(doc:Document)-[:APPLIES_TO]->(a:Aircraft)
        MATCH (a)-[:HAS_SYSTEM]->(s:System)
        OPTIONAL MATCH (s)-[:HAS_COMPONENT]->(comp:Component)
        WITH node, doc, a, s, comp, score
        RETURN doc.documentId AS document_id,
               doc.aircraftType AS aircraft_type,
               a.tail_number AS aircraft,
               COLLECT(DISTINCT s.name)[0..3] AS systems,
               COLLECT(DISTINCT comp.name)[0..3] AS components,
               substring(node.text, 0, 200) AS context,
               score
        ORDER BY score DESC
        LIMIT 3
    """
}))
print("Topology-enriched search:\n")
print(result)

**Graph traversal path:**
1. Fulltext search finds relevant maintenance chunks
2. `FROM_DOCUMENT` reaches the source Document
3. `APPLIES_TO` connects to the Aircraft
4. `HAS_SYSTEM` and `HAS_COMPONENT` walk the topology hierarchy

The result combines text context with structured aircraft data — systems, components — from a single query.

## Example 4: Operating Limits from Extracted Entities

Traverse the full chain: Chunk -> Document -> Aircraft -> System -> Sensor -> OperatingLimit.

In notebook 03, `SimpleKGPipeline` extracted `OperatingLimit` entities from maintenance text and cross-linked them to Sensor nodes. This query reaches those structured entities through graph traversal.

This mirrors notebook 04's `operating_limit_query` VectorCypherRetriever.

In [ ]:
result = asyncio.run(mcp_call_tool("read-cypher", {
    "query": """
        CALL db.index.fulltext.queryNodes('maintenanceChunkText', 'EGT temperature limits')
        YIELD node, score
        MATCH (node)-[:FROM_DOCUMENT]->(doc:Document)-[:APPLIES_TO]->(a:Aircraft)
        OPTIONAL MATCH (a)-[:HAS_SYSTEM]->(sys:System)-[:HAS_SENSOR]->(s:Sensor)-[:HAS_LIMIT]->(ol:OperatingLimit)
        WITH node, doc, a, score,
             COLLECT(DISTINCT {
                 sensor: s.type,
                 parameter: ol.parameterName,
                 max: ol.maxValue,
                 unit: ol.unit,
                 regime: ol.regime
             })[0..5] AS operating_limits
        RETURN doc.aircraftType AS aircraft_type,
               operating_limits,
               substring(node.text, 0, 200) AS context
        ORDER BY score DESC
        LIMIT 3
    """
}))
print("Operating limits from graph traversal:\n")
print(result)

**Full traversal chain:** Chunk -> Document -> Aircraft -> System -> Sensor -> OperatingLimit

This demonstrates the entity extraction payoff from notebook 03: the LLM extracted structured `OperatingLimit` entities from unstructured text, and those entities are now queryable through graph traversal via MCP.

> **Note:** The number of OperatingLimit entities depends on what the LLM extracted during the SimpleKGPipeline run in notebook 03. If entity extraction produced no OperatingLimit entities, the `operating_limits` field will be empty.

---

# Try Your Own Queries

Experiment with different Cypher queries through MCP. Some ideas:

- What are the vibration limits that require engine shutdown?
- How do I check for hydraulic fluid contamination?
- What oil analysis levels indicate bearing wear?
- When should I perform a borescope inspection?

In [ ]:
# Try your own query
result = asyncio.run(mcp_call_tool("read-cypher", {
    "query": """
        CALL db.index.fulltext.queryNodes('maintenanceChunkText', 'YOUR SEARCH TERMS HERE')
        YIELD node, score
        RETURN score, substring(node.text, 0, 300) AS context
        ORDER BY score DESC
        LIMIT 3
    """
}))
print(result)

---

# Comparing Approaches

| Aspect | Direct Driver (notebooks 04/05) | MCP (this notebook) |
|--------|------------------------|-------------|
| Connection | `neo4j://` driver with credentials | HTTPS + API key to MCP server |
| Authentication | Neo4j username/password | Bearer token (no DB creds exposed) |
| Dependencies | `neo4j`, `neo4j-graphrag` | `fastmcp` only |
| Search Entry Point | Vector search (embeddings) | Fulltext search (keywords) |
| Graph Traversal | `VectorCypherRetriever` abstraction | Explicit Cypher via `read-cypher` |
| Best For | Application code, GraphRAG pipelines | AI agents, tool-calling, decoupled access |

## Summary

In this notebook, you queried the same Neo4j knowledge graph from Labs 2-3 entirely through MCP:

**Part 1 - Discover & Explore:**
- Connected to an MCP server with FastMCP client and Bearer auth
- Discovered tools (`get-schema`, `read-cypher`) and retrieved the graph schema
- Ran basic Cypher queries for node counts and fleet overview

**Part 2 - Document-Chunk Queries:**
- Explored the Document-Chunk structure created in notebook 03
- Walked `NEXT_CHUNK` chains to see sequential chunk linking

**Part 3 - Graph-Enhanced Search:**
- Document context enrichment (fulltext search + `FROM_DOCUMENT`)
- Adjacent chunk retrieval (fulltext search + `NEXT_CHUNK`)
- Aircraft topology traversal (fulltext search + `APPLIES_TO` -> `HAS_SYSTEM` -> `HAS_COMPONENT`)
- Operating limits via full entity chain traversal

**Key takeaway:** MCP provides a standard protocol layer between your code and the database. The Cypher queries are identical to what runs inside the `VectorCypherRetriever` abstractions from notebook 04 — MCP just changes how they get to Neo4j.